In [88]:
import pandas as pd
import re 
import numpy as np
import nltk
from nltk.tokenize import word_tokenize
from nltk import pos_tag, ne_chunk
from nltk.corpus import cess_esp
import spacy
from nltk import CFG
import json

In [89]:
nltk.download('cess_esp')
try:
    nltk.data.find('tokenizers/punkt')
    print("El recurso 'punkt' se encuentra disponible.")
except LookupError:
    print("El recurso 'punkt' no fue encontrado. Intentando descargarlo nuevamente.")
    nltk.download('punkt')

tagger = nltk.UnigramTagger(nltk.corpus.cess_esp.tagged_sents())
nlp = spacy.load('es_core_news_sm')

[nltk_data] Downloading package cess_esp to
[nltk_data]     C:\Users\ma907\AppData\Roaming\nltk_data...
[nltk_data]   Package cess_esp is already up-to-date!


El recurso 'punkt' se encuentra disponible.


In [90]:
data = pd.read_csv("results---From---2023-10-31--22-08-07---To---2024-04-07--16-02-28.csv")
data = data[['message']].dropna()
data = data.drop_duplicates()
data

,message
0,#Renta x DÍAS de Apto en el Vedado
1,No tienes permisos para ejecutar este comando ...
2,/revisarbrplus@ReputacionPlusBot
8,Casa en venta en la zona sur cerca de las fábr...
9,Busco renta por tiempo indefinido para una par...
...,...
4988,Busco alquiler en el vedado límite 150 verde s...
4992,"Busco alquiler por tiempo indefinido, 58316712"
4993,Busco alquiler en la lisa o lo más cerca posible
4994,Busco alquiler en la Lisa


### Limpiando los datos para poder empezar a trabajar sobre ellos

In [91]:
def delete_emojis(text):
    patron_emojis = re.compile(pattern="["
                                      u"\U0001F600-\U0001F64F"  
                                      u"\U0001F300-\U0001F5FF"  
                                      u"\U0001F680-\U0001F6FF"  
                                      u"\U0001F700-\U0001F77F"  
                                      u"\U0001F780-\U0001F7FF"  
                                      u"\U0001F800-\U0001F8FF"  
                                      u"\U0001F900-\U0001F9FF"  
                                      u"\U0001FA00-\U0001FAFF" 
                                      u"\U00002702-\U000027B0"  
                                      u"\U00002702-\U000027B0"
                                      u"\U000024C2-\U0001F251"
                                      "]+", flags=re.UNICODE)
    return patron_emojis.sub(r'', text)

def delete_commands(texto):
    cleaned_text = re.sub(r'/\S+', '', texto)
    cleaned_text = re.sub(r'@\S+', '', cleaned_text)
    cleaned_text = re.sub(r'#', '', cleaned_text)
    cleaned_text = re.sub(r'http[s]?://\S+', '', cleaned_text)
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()
    
    return cleaned_text

def normalizar_texto(texto):
    return texto.lower()

#def tokenizar(texto):
 #   return word_tokenize(texto)

def tokenizar(texto):
    doc = nlp(texto)
    return [token.text for token in doc]

def etiquetar_pos_spacy(texto):
    doc = nlp(texto)
    return [(token.text, token.pos_) for token in doc]

def lemmatize(texto):
    doc = nlp(texto)
    lemmatized_text = ' '.join([token.lemma_ for token in doc])
    
    return lemmatized_text

def extraer_entidades(texto):
    doc = nlp(texto)
    return [(ent.text, ent.label_) for ent in doc.ents]

def preprocesar_texto(texto):
    texto = delete_emojis(texto)
    texto = delete_commands(texto)
    tokens = tokenizar(texto)
    texto_procesado = ' '.join(tokens)
    pos_tags = etiquetar_pos_spacy(texto_procesado)
    return pos_tags

#### Ahora vamos a empezar a extraer features de interés y vamos empezar con el precio y la moneda en que se haria la negociación

In [92]:
def extract_price(message):
    prices = re.findall(r'\b\d{2,6}(?:[.,]\d+)? | \d{2,6}(?:[.,]\d+)? mil\b', message)
    if prices:
        return prices
    return None


def extract_currency(message):
    currencies = re.findall(r'\b(USD|usd|dólar|dolar|EURO|euro|MLC|mlc|CUP|cup|pesos|mn|dolar|dolares|mil)\b', message, re.IGNORECASE)
    if currencies:
        return currencies
    return None

##### Además del precio de los alquileres necesito extraer la ubicación de los mismos, para ello voy a utilizar el modelo NER preentrenado de spacy(poco a poco lo iré mejorando)

In [93]:
def extract_location(message):
    doc = nlp(message)
    location = [ent.text for ent in doc.ents if ent.label_ == 'LOC']
    return location

data= data.dropna()
print(data)

                                                message
0                    #Renta x DÍAS de Apto en el Vedado
1     No tienes permisos para ejecutar este comando ...
2                      /revisarbrplus@ReputacionPlusBot
8     Casa en venta en la zona sur cerca de las fábr...
9     Busco renta por tiempo indefinido para una par...
...                                                 ...
4988  Busco alquiler en el vedado límite 150 verde s...
4992     Busco alquiler por tiempo indefinido, 58316712
4993   Busco alquiler en la lisa o lo más cerca posible
4994                          Busco alquiler en la Lisa
4996  Busco alquiler en playa, Marianao, lisa hasta ...

[2508 rows x 1 columns]


##### Vamos a ir probando

In [94]:
data['message'] = data['message'].apply(delete_emojis)
data['message'] = data['message'].apply(delete_commands) 
data['message'] = data['message'].apply(normalizar_texto)
#data['tokens'] = data['message'].apply(tokenizar)
#data['entidades'] = data['message'].apply(extract_location)
data['precio'] = data['message'].apply(extract_price)
data['moneda'] = data['message'].apply(extract_currency) 

print(data)


                                                message
0                     renta x días de apto en el vedado
1     no tienes permisos para ejecutar este comando ...
2                                                      
8     casa en venta en la zona sur cerca de las fábr...
9     busco renta por tiempo indefinido para una par...
...                                                 ...
4988  busco alquiler en el vedado límite 150 verde s...
4992     busco alquiler por tiempo indefinido, 58316712
4993   busco alquiler en la lisa o lo más cerca posible
4994                          busco alquiler en la lisa
4996  busco alquiler en playa, marianao, lisa hasta ...

[2508 rows x 1 columns]


##### Bueno, ya probe utilizar NER para detectar las ubicaciones, sin embargo, los modelos prentrenados de Spacy y de NLTK no son buenos detectando ubicaciones tan especificas. Por consiguiente para empezar vamos a utilizar regex y cfg para ver cual es mejor para esto y vamos a ver que pasa.

In [95]:
with open("barrios_calles_habana.json", "r", encoding="utf-8") as file:
    json_data = json.load(file)

streets = [r"(calle\s+\w+(?:\s+\w+)*|calzada\s+\w+(?:\s+\w+)*)"]
intersections = r"((entre|esquina|y)\s+(calle|calzada|av\.\?|avenida)\s+\w+(?:\s+\w+)*)"
neighborhoods = [r"(centro Habana|vedado|boyeros|playa|marianao|bahía|miramar)"]
municipalities = [r"(10 de Octubre|san Miguel|habana del este|santos suárez)"]
nearby = r"(cerca de\s+\w+(?:\s+\w+)*)"
landmarks = r"(Terminal de Ómnibus Nacionales|Plaza de la Revolución|Calixto García|Pediátrico de Centro Habana|Fajardo|Ciudad Deportiva)"

for municipality, areas in json_data.items():
    municipalities.append(re.escape(municipality.lower())) 

    for neighborhood, streets_list in areas.items():
        neighborhoods.append(re.escape(neighborhood.lower())) 
        for street in streets_list:
            streets.append(re.escape(street.lower()))

streets_pattern = "|".join(streets)
neighborhoods_pattern = "|".join(neighborhoods)
municipalities_pattern = "|".join(municipalities)

location_pattern = rf"\b({streets_pattern}|{intersections}|{neighborhoods_pattern}|{municipalities_pattern}|{nearby}|{landmarks})\b"

def find_locations(message):
    matches = re.findall(location_pattern, message, re.IGNORECASE)
    unique_matches = list(set([match[0].strip() if isinstance(match, tuple) else match.strip() for match in matches]))
    return sorted(unique_matches)

data['ubicaciones'] = data['message'].apply(find_locations)

data

,message,ubicaciones
0,renta x días de apto en el vedado,[el vedado]
1,no tienes permisos para ejecutar este comando ...,[]
2,,[]
8,casa en venta en la zona sur cerca de las fábr...,"[cerca de las fábricas de cerveza y galleta, h..."
9,busco renta por tiempo indefinido para una par...,[]
...,...,...
4988,busco alquiler en el vedado límite 150 verde s...,[el vedado]
4992,"busco alquiler por tiempo indefinido, 58316712",[]
4993,busco alquiler en la lisa o lo más cerca posible,[la lisa]
4994,busco alquiler en la lisa,[la lisa]


##### Usando CFG